# PK-PD simulator validation

Standalone CPU validation of the research-only Schnider/Minto/Yun PK–PD reconstruction. This is not a medical device and must not be used for clinical dosing. No RL training is performed.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CONTENT_ROOT = Path('/content')
REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = CONTENT_ROOT / 'VitalDB-Feature-Selection'
OUTPUT_DIR = CONTENT_ROOT / 'pkpd_simulator_validation'

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository HEAD:', subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scipy', 'matplotlib'], check=True)
print('CPU validation dependencies are ready.')

In [ ]:
command = [
    sys.executable, '-u', 'scripts/run_pkpd_simulator_validation.py',
    '--patient', 'middle_male',
    '--duration-seconds', '1800',
    '--internal-dt-seconds', '1',
    '--sample-interval-seconds', '10',
    '--integrator', 'exact',
    '--propofol-schedule', 'induction_maintenance_recovery',
    '--remifentanil-schedule', 'piecewise',
    '--seed', '20260716',
    '--output-dir', str(OUTPUT_DIR),
]
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

for name in ['reference_patient_parameters.csv', 'trajectory.csv', 'integrator_comparison.csv']:
    display(Markdown(f'## {name}'))
    frame = pd.read_csv(OUTPUT_DIR / name)
    display(frame if len(frame) <= 20 else frame.head(20))

display(Markdown('## Validation summary'))
display(json.loads((OUTPUT_DIR / 'validation_summary.json').read_text()))
display(Markdown('## Simulator manifest'))
display(json.loads((OUTPUT_DIR / 'simulator_manifest.json').read_text()))

In [ ]:
for name in [
    'propofol_cp_ce.png', 'remifentanil_cp_ce.png', 'bis_trajectory.png',
    'infusion_and_bis.png', 'recovery_trajectory.png',
]:
    display(Markdown(f'## {name}'))
    display(Image(filename=str(OUTPUT_DIR / 'figures' / name)))

display(Markdown((OUTPUT_DIR / 'pkpd_validation_report.md').read_text()))